# 00 — Quickstart: the v2 WRDS/R2 research spine

**The project in one paragraph.** We buy corporate bonds just downgraded from
investment-grade to high-yield ("fallen angels"), hold roughly six months, and earn the
rebound. The economic claim: IG-mandated holders (insurers, pensions, IG-only funds,
index trackers) are forced to sell on the downgrade *because the label changed*, and that
concentrated selling pushes the price below fair value temporarily. Part I of the
methodology proves (or falsifies) that the dislocation is real and mechanism-driven;
Part II turns it into a risk-managed, hedged, cost-aware portfolio. Nothing in Part II is
trusted until Part I holds.

**Read before doing anything:**

| Doc | What it governs |
|---|---|
| `CLAUDE.md` | Agent guide — what exists, the invariants, the repo map |
| `methodology_finance_economics_math.md` | The "bible" — every research/modelling decision |
| `docs/roadmap/` | The 20-step build plan — find the current step, stay in scope |
| `docs/architecture/pipeline_v2.md` | v2 module contracts — signatures, schemas, retirements |
| `docs/R2_data_dictionary/` | Ground-truth schema dump of the WRDS/R2 mirror |

This notebook is an onboarding smoke test — verify credentials, probe the mirror the
right way, read a derived artifact — not a tutorial.

## 1. Install dependencies
Run once in your environment:

In [ ]:
# !pip install duckdb pandas matplotlib python-dotenv

## 2. Set credentials
Create a `.env` file in this directory (never commit it):

In [ ]:
# .env template — canonical names (match config/.env and src/data/r2.py).
# The shorter aliases R2_KEY_ID / R2_SECRET are also accepted by the library.
# R2_ENDPOINT=https://<ACCOUNT_ID>.r2.cloudflarestorage.com
# R2_ACCESS_KEY_ID=<YOUR_READ_ONLY_KEY_ID>
# R2_SECRET_ACCESS_KEY=<YOUR_READ_ONLY_SECRET>
# R2_BUCKET=quantt-historical-market-data

## 3. Connect

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
from dotenv import load_dotenv

# Load config/.env if present (repo keeps credentials there), else a local .env.
load_dotenv("config/.env")
load_dotenv()


def _env(*names, default=None):
    """First non-empty env var among names (accepts canonical + alias names)."""
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return default


ENDPOINT = _env("R2_ENDPOINT")
KEY_ID   = _env("R2_ACCESS_KEY_ID", "R2_KEY_ID")
SECRET   = _env("R2_SECRET_ACCESS_KEY", "R2_SECRET")
BUCKET   = _env("R2_BUCKET", default="quantt-historical-market-data")
assert ENDPOINT and KEY_ID and SECRET, "Set R2 credentials in config/.env (see 00 cell above)."

con = duckdb.connect()
con.execute("""
    INSTALL httpfs; LOAD httpfs;
    SET s3_endpoint    = '{endpoint}';
    SET s3_access_key_id     = '{key_id}';
    SET s3_secret_access_key = '{secret}';
    SET s3_region = 'auto';
    SET s3_url_style = 'path';
""".format(
    endpoint = ENDPOINT.replace('https://', ''),
    key_id   = KEY_ID,
    secret   = SECRET,
))


def r2(schema, table):
    """Return the S3 path for a given schema/table."""
    return f"s3://{BUCKET}/wrds/{schema}/{table}.parquet"

def q(sql):
    return con.execute(sql).df()

print("Connected to R2.")

## 4. Test query

In [ ]:
result = q(f"SELECT * FROM read_parquet('{r2('ff_all', 'factors_daily')}') LIMIT 3")
print(result)
print("Connection working! Fama-French daily factors:")
print(result.columns.tolist())